# NB18 — Bayesian Linear Regression

> **Instead of a single point estimate, get a full posterior distribution over coefficients.**

Frequentist OLS: β̂ is a fixed (unknown) constant. We estimate it.
Bayesian: β is a **random variable** with a prior distribution. Data updates it to a posterior.

```
posterior ∝ likelihood × prior
p(β | data) ∝ p(data | β) × p(β)
```

With a Gaussian prior on β and Gaussian likelihood (normal noise), the posterior is also Gaussian — closed-form solution.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
# True parameters
true_b0, true_b1 = 3.0, 2.0
sigma_noise = 2.0

# Generate data
n = 30
X = np.random.uniform(0, 5, n)
y = true_b0 + true_b1 * X + np.random.normal(0, sigma_noise, n)

# Bayesian linear regression with conjugate Gaussian prior
# Prior: β ~ N(μ₀, Σ₀)
# Posterior: β | y, X ~ N(μₙ, Σₙ)

def bayesian_lr(X, y, sigma2_noise, mu_0, Sigma_0_inv):
    n = len(X)
    X_d = np.column_stack([np.ones(n), X])   # design matrix
    Sigma_n_inv = Sigma_0_inv + X_d.T @ X_d / sigma2_noise
    Sigma_n     = np.linalg.inv(Sigma_n_inv)
    mu_n        = Sigma_n @ (Sigma_0_inv @ mu_0 + X_d.T @ y / sigma2_noise)
    return mu_n, Sigma_n

# Weakly informative prior: β ~ N([0,0], 10I)
mu_0          = np.array([0., 0.])
Sigma_0_inv   = np.eye(2) / 10.0

mu_n, Sigma_n = bayesian_lr(X, y, sigma_noise**2, mu_0, Sigma_0_inv)

print(f"True parameters:     β₀={true_b0:.2f},  β₁={true_b1:.2f}")
print(f"OLS estimates:       β₀={np.polyfit(X,y,1)[1]:.4f},  β₁={np.polyfit(X,y,1)[0]:.4f}")
print(f"Posterior mean (MAP):β₀={mu_n[0]:.4f},  β₁={mu_n[1]:.4f}")
print(f"Posterior std:       β₀={np.sqrt(Sigma_n[0,0]):.4f},  β₁={np.sqrt(Sigma_n[1,1]):.4f}")


In [ ]:
# Show how posterior updates as data arrives
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
X_plot = np.linspace(0, 5, 200)

for ax, n_obs in zip(axes, [3, 10, 30]):
    X_sub = X[:n_obs]; y_sub = y[:n_obs]
    mu_sub, Sig_sub = bayesian_lr(X_sub, y_sub, sigma_noise**2, mu_0, Sigma_0_inv)

    # Sample 50 lines from the posterior
    samples = np.random.multivariate_normal(mu_sub, Sig_sub, 50)
    ax.scatter(X_sub, y_sub, color='steelblue', s=30, zorder=3, label='data')
    for s in samples:
        ax.plot(X_plot, s[0] + s[1]*X_plot, 'r-', alpha=0.08, linewidth=1)
    ax.plot(X_plot, mu_sub[0] + mu_sub[1]*X_plot, 'k-', linewidth=2.5, label='Posterior mean')
    ax.plot(X_plot, true_b0 + true_b1*X_plot, 'g--', linewidth=1.5, label='True line')
    ax.set_title(f'n={n_obs} observations'); ax.legend(fontsize=7); ax.grid(alpha=0.3)

plt.suptitle('Bayesian regression: uncertainty shrinks as data grows', fontsize=12)
plt.tight_layout(); plt.show()


In [ ]:
# Posterior predictive distribution — gives prediction intervals automatically
import numpy as np
import matplotlib.pyplot as plt

X_new = np.linspace(0, 6, 100)   # include extrapolation
mu_n_f, Sigma_n_f = bayesian_lr(X, y, sigma_noise**2, mu_0, Sigma_0_inv)

pred_mean = mu_n_f[0] + mu_n_f[1] * X_new

# Predictive variance = uncertainty in f(x) + measurement noise
n_full = len(X)
X_full = np.column_stack([np.ones(n_full), X])
Xnew_d = np.column_stack([np.ones(len(X_new)), X_new])
pred_var = np.array([Xnew_d[i] @ Sigma_n_f @ Xnew_d[i] + sigma_noise**2
                     for i in range(len(X_new))])
pred_std = np.sqrt(pred_var)

plt.figure(figsize=(10, 5))
plt.scatter(X, y, color='steelblue', s=30, zorder=3, label='Observed data')
plt.plot(X_new, pred_mean, 'r-', linewidth=2, label='Predictive mean')
plt.fill_between(X_new, pred_mean-2*pred_std, pred_mean+2*pred_std,
                 alpha=0.2, color='orange', label='95% predictive interval')
plt.axvline(X.max(), color='gray', linewidth=1, linestyle='--', label='Extrapolation boundary')
plt.xlabel('X'); plt.ylabel('y')
plt.title('Bayesian predictive interval — wider in extrapolation region')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


## Key Takeaways

| | OLS | Bayesian |
|--|-----|---------|
| Output | Point estimate β̂ | Full distribution p(β|data) |
| Prior knowledge | Not used | Incorporated as prior |
| Uncertainty | From t-distribution | From posterior |
| Small n | Unreliable | Prior regularises |
| Closed form | Yes (normal eqns) | Yes (conjugate Gaussian) |

With a Gaussian prior and `σ² = 1/λ`, Bayesian MAP = Ridge regression.

**Next:** NB19 — Linear regression for time series.
